<a href="https://colab.research.google.com/github/sina-04/dental-yolo26-detection/blob/main/notebooks/dental_yolo26_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dental YOLO26 — pathology-first A100/L4 workflow

This staged notebook preserves the original 31-class benchmark and creates a primary six-class pathology view for Caries, Periapical lesion, Retained root, Root Piece, impacted tooth, and Bone Loss. It reconstructs leakage-safe 70/15/15 patient-grouped splits, audits independent-patient support, compares 640/1024-pixel YOLO26s/m models, tunes only on validation data, and evaluates the held-out test set once.

In [ ]:
from google.colab import drive
import torch

drive.mount('/content/drive')
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > GPU, then reconnect.'
print(torch.cuda.get_device_name(0))
print(f'{torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GiB VRAM')


In [ ]:
from pathlib import Path

repo = Path('/content/dental-yolo26-detection')
if (repo / '.git').exists():
    !git -C {repo} pull --ff-only
else:
    !git clone https://github.com/sina-04/dental-yolo26-detection.git {repo}
%cd /content/dental-yolo26-detection
!python -m pip install -q -r requirements-colab.txt
!python -m unittest discover -s tests -v


In [ ]:
# Use the exploratory profile with the public dataset. Change to the strict
# pathology_a100.yaml only after every support requirement is satisfied.
PROFILE = 'configs/pathology_a100_exploratory.yaml'
RESULTS_ROOT = '/content/drive/MyDrive/dental-yolo26-detection/pathology6-a100-v2'


## 1. Prepare and verify

The exact Kaggle v6 export is downloaded to ephemeral SSD. Preparation creates the unchanged 31-class dataset and a disk-efficient remapped pathology view. Raw images are not copied to Drive. Remove `--rebuild-data` on later runs to reuse an already prepared dataset in the same Colab session.

In [ ]:
!python -m src.colab_workflow \
  --stage prepare \
  --data-root /content/dental_yolo26_data \
  --rebuild-data


In [ ]:
import json
support = json.loads(Path('reports/pathology_support.json').read_text())
print(json.dumps(support, indent=2))
if not support['support_gate_passed']:
    print('EXPLORATORY ONLY: collect independent examinations before using the strict final profile.')


## 2. Required annotation audit

Inspect every sheet below. Red boxes are the target class; blue boxes are other labeled findings. Training cannot start without an approval tied to this dataset fingerprint. Approval records review completion—not clinical validation.

In [ ]:
from IPython.display import Image, display
from pathlib import Path

sheets = sorted(Path('reports/annotation_audit').glob('class_*.jpg'))
assert len(sheets) == 31, f'Expected 31 sheets, found {len(sheets)}'
for sheet in sheets:
    print(sheet.name)
    display(Image(filename=str(sheet), width=740))


In [ ]:
import subprocess, sys

AUDIT_APPROVED = False  # Set True only after reviewing all 31 sheets above.
REVIEWER = 'Your name'
NOTES = 'Summarize label quality findings and any questionable classes.'
assert AUDIT_APPROVED, 'Review every contact sheet, complete REVIEWER/NOTES, then set AUDIT_APPROVED=True.'
subprocess.run([sys.executable, '-m', 'src.approve_audit', '--reviewer', REVIEWER, '--notes', NOTES], check=True)


In [ ]:
# Required by the strict final profile; do not self-certify unless qualified.
CLINICIAN_REVIEW_COMPLETE = False
CLINICIAN = 'Qualified dental reviewer'
CREDENTIALS = 'Professional credentials'
CLINICAL_NOTES = 'Summary of validation/test annotation and mined-error review.'
if CLINICIAN_REVIEW_COMPLETE:
    subprocess.run([sys.executable, '-m', 'src.approve_pathology_review', '--reviewer', CLINICIAN, '--credentials', CREDENTIALS, '--notes', CLINICAL_NOTES, '--confirm-complete'], check=True)
else:
    print('Clinician review not recorded; use the exploratory profile only.')


## 3. Train and select on validation data

The profile reproduces the 640-pixel baseline, tests YOLO26s and YOLO26m at 1024 px, and adds a patient-balanced training view. Runs use up to 120 epochs with 30-epoch early stopping. Use an A100; an L4 may require reducing profile batch sizes.

In [ ]:
!python -m src.colab_workflow \
  --stage train \
  --profile {PROFILE} \
  --results-root {RESULTS_ROOT}


## 4. Tune on validation data

The strict A100 profile runs 24 short randomized trials, fully trains the best three configurations, and confirms the winner across seeds 42, 43, and 44. The exploratory profile intentionally omits this expensive tuning section; use the strict profile after collecting enough independent patients.

In [ ]:
if PROFILE.endswith('pathology_a100.yaml'):
    !python -m src.colab_workflow --stage tune --profile {PROFILE} --results-root {RESULTS_ROOT}
else:
    print('Skipping final tuning: exploratory profile is under the patient-support gate.')


## 5. One-time held-out test evaluation

Run only after the validation-selected experiment is final. A matching completed result is reused. Do not pass `--force-test` merely to inspect or tune against test results.

In [ ]:
selection_path = Path(RESULTS_ROOT) / 'reports' / 'selection.json'
if not selection_path.exists():
    print('Training/selection is still in progress. Run this cell after the training cell completes.')
else:
    !python -m src.colab_workflow \
      --stage test \
      --profile {PROFILE} \
      --results-root {RESULTS_ROOT}


## 6. Generate deliverable reports

In [ ]:
results = Path(RESULTS_ROOT)
metrics_path = results / 'reports' / 'final_metrics.json'
if not metrics_path.exists():
    print('Held-out test metrics are not ready. Complete the training and test cells first.')
else:
    !python -m src.colab_workflow \
      --stage report \
      --results-root {RESULTS_ROOT}
    print('Best model:', results / 'artifacts/best.pt')
    print('Metrics:', metrics_path)
    print('Report:', results / 'FINAL_REPORT.md')
    print('Progress:', results / 'PROGRESS.md')
    assert (results / 'artifacts/best.pt').exists()
